<a href="https://colab.research.google.com/github/suraj76543/ViT-Llama-Latex-code-generator-/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch pillow datasets

In [ ]:
!pip install --no-deps unsloth
!pip install bitsandbytes accelerate peft trl triton cut_cross_entropy
!pip install sentencepiece protobuf datasets huggingface_hub hf_transfer


In [ ]:
!pip install unsloth_zoo
from unsloth import FastVisionModel
import torch

In [ ]:
fourbit_models = [
    "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit",
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit"
]

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen2-VL-7B-Instruct-bnb-4bit",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
    device_map="auto",
)


In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,

    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state = 3407,
    use_rslora=False,
    loftq_config=None
)

In [ ]:
from torch.utils.data import Dataset


In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import os

class IM2LatexDataset(Dataset):
    def __init__(self, root_dir):
        self.root_dir = root_dir

        # auto-detect txt location
        if os.path.exists(os.path.join(root_dir, "corresponding_png_images.txt")):
            base = root_dir
        elif os.path.exists(os.path.join(root_dir, "data", "corresponding_png_images.txt")):
            base = os.path.join(root_dir, "data")
        else:
            raise FileNotFoundError("Could not find corresponding_png_images.txt")

        self.base = base

        with open(os.path.join(base, "corresponding_png_images.txt")) as f:
            self.image_files = [l.strip() for l in f]

        with open(os.path.join(base, "final_png_formulas.txt")) as f:
            self.formulas = [l.strip() for l in f]

        assert len(self.image_files) == len(self.formulas)

    def __len__(self):
        return len(self.formulas)

    def __getitem__(self, idx):
        image_path = os.path.join(
            self.base,
            "generated_png_images",
            self.image_files[idx]
        )
        image = Image.open(image_path).convert("RGB")

        return {
            "image": image,
            "formula": self.formulas[idx],
        }


In [ ]:
!mkdir -p /content/datasets


In [ ]:
!unzip -q "/content/drive/MyDrive/IMG2LaTeX Zip.zip" -d /content/datasets


In [ ]:
!ls /content/datasets


In [ ]:
!ls /content/datasets/PRINTED_TEX_230k


In [ ]:
root_dir = "/content/datasets/PRINTED_TEX_230k"


In [ ]:
dataset = IM2LatexDataset("/content/datasets/PRINTED_TEX_230k")
dataset[0]


In [ ]:
instruction = "Write the LaTeX representation for this image."

def convert_to_conversation(sample):
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": instruction},
                    {"type": "image", "image": sample["image"]}
                ],
            },
            {
                "role": "assistant",
                "content": [
                    {"type": "text", "text": sample["formula"]}
                ],
            },
        ]
    }


In [ ]:
sample = dataset[0]
conv = convert_to_conversation(sample)
conv


In [ ]:
train_size = 100
test_size  = 20

train_dataset = [convert_to_conversation(dataset[i]) for i in range(train_size)]
test_dataset  = [
    convert_to_conversation(dataset[i])
    for i in range(train_size, train_size + test_size)
]


In [ ]:
from unsloth import UnslothVisionDataCollator


In [ ]:
data_collator = UnslothVisionDataCollator(
    model = model,
    processor = tokenizer
)


In [ ]:
from transformers import TrainingArguments, Trainer


In [ ]:
training_args = TrainingArguments(
    output_dir="./vision2tex",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-5,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=200,
    fp16=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,   # make sure this is correct
    data_collator=data_collator,
)


In [ ]:
trainer.train()


In [ ]:
FastVisionModel.for_inference(model)

sample = dataset[1]

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": instruction},
        ],
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors="pt"
)

inputs = tokenizer(
    images=sample["image"],
    text=inputs,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=False,
)

decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

# extract only the assistant's LaTeX part
latex = decoded.split("assistant")[-1].strip()

# remove $$ if present
latex = latex.replace("$$", "").strip()

print(latex)



In [ ]:
def image_to_latex(image):
    FastVisionModel.for_inference(model)

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": instruction},
            ],
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = tokenizer(
        images=image,
        text=prompt,
        return_tensors="pt"
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
!find /content/datasets -name "000a20147419b28.png"


In [ ]:
from PIL import Image
from IPython.display import display

# Using an image from the loaded dataset for demonstration
image = dataset[0]["image"]
display(image)

In [ ]:
latex_output = image_to_latex(image)
latex_output = latex_output.split("assistant")[-1]
latex_output = latex_output.replace("$$", "").strip()
print(latex_output)


In [ ]:
model.save_pretrained("model_save")
tokenizer.save_pretrained("model_save")

In [ ]:
# model, tokenizer = FastVisionModel.from_pretrained(
#     "model_save",
#     load_in_4bit=True,
#     use_gradient_checkpointing="unsloth",
#     device_map="auto",
# )

In [ ]:
# import re

# def normalize_latex(tex):
#     tex = tex.lower()
#     tex = tex.replace("$$", "")
#     tex = tex.replace("\\,", "")
#     tex = tex.replace("\\!", "")
#     tex = tex.replace("\\left", "").replace("\\right", "")
#     tex = re.sub(r"\s+", "", tex)     # remove spaces
#     tex = re.sub(r"[{}]", "", tex)    # remove braces
#     return tex


In [ ]:
# preds = []
# refs  = []

# for i in range(len(test_dataset)):
#     sample = test_dataset[i]

#     image = sample["messages"][0]["content"][1]["image"]
#     gt    = sample["messages"][1]["content"][0]["text"]

#     pred = image_to_latex(image)

#     preds.append(normalize_latex(pred))
#     refs.append(normalize_latex(gt))


In [ ]:
# !pip install -q sacrebleu


In [ ]:
# !pip install -q python-Levenshtein


In [ ]:
# # 1. Exact Match Accuracy (EMA)
# def exact_match_accuracy(preds, refs):
#     return sum(p == r for p, r in zip(preds, refs)) / len(refs)


# # 2. Expression Recognition Rate (ERR)
# def normalize_latex(tex):
#     tex = tex.replace(" ", "")
#     tex = tex.replace("{", "").replace("}", "")
#     tex = tex.replace("\\left", "").replace("\\right", "")
#     return tex

# def expression_recognition_rate(preds, refs):
#     return sum(
#         normalize_latex(p) == normalize_latex(r)
#         for p, r in zip(preds, refs)
#     ) / len(refs)


# # 3. BLEU score
# import sacrebleu

# def bleu_score(preds, refs):
#     refs = [[r] for r in refs]
#     return sacrebleu.corpus_bleu(preds, refs).score


# # 4. Normalized Edit Distance (NED)
# import Levenshtein
# import numpy as np

# def normalized_edit_distance(preds, refs):
#     return np.mean([
#         Levenshtein.distance(p, r) / max(len(r), 1)
#         for p, r in zip(preds, refs)
#     ])


In [ ]:
# ema  = exact_match_accuracy(preds, refs)
# err  = expression_recognition_rate(preds, refs)
# bleu = bleu_score(preds, refs)
# ned  = normalized_edit_distance(preds, refs)

# print(f"Exact Match Accuracy (EMA): {ema:.4f}")
# print(f"Expression Recognition Rate (ERR): {err:.4f}")
# print(f"BLEU Score: {bleu:.2f}")
# print(f"Normalized Edit Distance (NED): {ned:.4f}")
